# Exploration — Occupants et socio-economique
Analyse du vintage (decennie de construction), du profil des occupants et impact sur la consommation — ResStock 2025

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')
CONSO_COL     = 'out.electricity.total.energy_consumption..kwh'
INTENSITY_COL = 'out.electricity.total.energy_consumption_intensity..kwh_per_ft2'
df[CONSO_COL]     = pd.to_numeric(df[CONSO_COL], errors='coerce')
df[INTENSITY_COL] = pd.to_numeric(df[INTENSITY_COL], errors='coerce')
print('Shape:', df.shape)

## 1. Vintage — Timeline decennale

Deux lectures superposees :
- **Barres grises** : poids de chaque decennie dans le dataset
- **Ligne rouge** : consommation totale moyenne (kWh/an)
- **Ligne orange pointillee** : intensite (kWh/ft2) — mesure l efficacite independamment de la taille

In [ ]:
VINTAGE_ORDER = ['<1940','1940s','1950s','1960s','1970s','1980s','1990s','2000s','2010s']

grp = df.groupby('in.vintage').agg(
    count=('in.vintage', 'count'),
    conso_moy=(CONSO_COL, 'mean'),
    intensity_moy=(INTENSITY_COL, 'mean'),
).reindex(VINTAGE_ORDER)

x = list(range(len(VINTAGE_ORDER)))

fig, ax1 = plt.subplots(figsize=(13, 6))

# Barres = nb batiments
ax1.bar(x, grp['count'], color='#bdc3c7', alpha=0.8, label='Nb batiments')
ax1.set_xticks(x)
ax1.set_xticklabels(VINTAGE_ORDER, fontsize=11)
ax1.set_ylabel('Nombre de batiments', color='#7f8c8d', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#7f8c8d')
ax1.set_xlabel('Decennie de construction', fontsize=11)

# Axe secondaire : conso totale + intensite
ax2 = ax1.twinx()
l1, = ax2.plot(x, grp['conso_moy'], color='#e74c3c', marker='o',
               linewidth=2.5, markersize=8, label='Conso moy. (kWh/an)')
l2, = ax2.plot(x, grp['intensity_moy'] * 1000, color='#e67e22', marker='s',
               linewidth=2, markersize=7, linestyle='--',
               label='Intensite x1000 (kWh/ft2)')
ax2.set_ylabel('kWh/an  |  Intensite x1000 kWh/ft2', color='#c0392b', fontsize=9)
ax2.tick_params(axis='y', labelcolor='#c0392b')

idx_max = grp['conso_moy'].idxmax()
idx_min = grp['conso_moy'].idxmin()
ax2.annotate(f"{grp.loc[idx_max,'conso_moy']:.0f} kWh",
             xy=(VINTAGE_ORDER.index(idx_max), grp.loc[idx_max,'conso_moy']),
             xytext=(8, 6), textcoords='offset points', fontsize=8, color='#e74c3c')
ax2.annotate(f"{grp.loc[idx_min,'conso_moy']:.0f} kWh",
             xy=(VINTAGE_ORDER.index(idx_min), grp.loc[idx_min,'conso_moy']),
             xytext=(8, -14), textcoords='offset points', fontsize=8, color='#e74c3c')

bar_handle = plt.Rectangle((0,0),1,1, color='#bdc3c7', alpha=0.8)
ax1.legend(handles=[bar_handle, l1, l2],
           labels=['Nb batiments','Conso moy. (kWh/an)','Intensite x1000 (kWh/ft2)'],
           loc='upper left', fontsize=9)

fig.suptitle(
    "Batiments par decennie de construction\n"
    "Paradoxe : plus recent = plus consommateur en absolu, mais plus efficace par ft2",
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(FIGURES / 'vintage_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Max conso  : {idx_max} ({grp.loc[idx_max,"conso_moy"]:.0f} kWh/an)')
print(f'Min conso  : {idx_min} ({grp.loc[idx_min,"conso_moy"]:.0f} kWh/an)')
print(f'Intensite min : {grp["intensity_moy"].idxmin()} ({grp["intensity_moy"].min():.1f} kWh/ft2)')
print('Sauvegarde : vintage_timeline.png')

## 2. Nombre d occupants — Distribution et consommation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution du nombre d occupants
occ = df['in.occupants'].value_counts().sort_index()
axes[0].bar(occ.index.astype(str), occ.values, color='#5b9bd5', alpha=0.8)
axes[0].set_xlabel('Nombre d occupants')
axes[0].set_ylabel('Nb batiments')
axes[0].set_title('Distribution des occupants', fontweight='bold')
for bar, val in zip(axes[0].patches, occ.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 str(val), ha='center', fontsize=7)

# Conso moy par nombre d occupants
df['occ_num'] = pd.to_numeric(df['in.occupants'], errors='coerce')
conso_occ = df.groupby('occ_num')[CONSO_COL].mean().dropna()
axes[1].bar(conso_occ.index.astype(str), conso_occ.values, color='#e74c3c', alpha=0.8)
axes[1].set_xlabel('Nombre d occupants')
axes[1].set_ylabel('Conso moyenne (kWh/an)')
axes[1].set_title('Consommation moyenne par nb d occupants', fontweight='bold')

plt.suptitle('Groupe Occupants — Distribution et impact sur la consommation',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'occupants_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegarde : occupants_distribution.png')